In [ ]:
from ruamel import yaml

# load YAML analysis "manifest", i.e. list of samples to include 
# plus some metadata to clearly label experimental conditions, e.g. cell type
with open('myc-evi_analysis_manifest_gabi-samples.yaml') as fd:
    YAML = yaml.YAML()
    res = YAML.load(fd)
res

In [ ]:
from pathlib import Path
import pandas as pd

def load_rprop_file_add_sample_info(rprop_file, sample_info):
    df_i = pd.read_csv(rprop_file)

    # add experimental metadata
    df_i['sample'] = sample_info['sample_name']
    df_i['batch'] = sample_info['batch_name']
    df_i['cell_type'] = sample_info['cell_type']

    return df_i

def load_all_rprop_files_for_sample(sample_info):
    pattern = sample_info["subsample_pattern"] if sample_info["subsample_pattern"] is not None else ""
    rprop_files = sorted((Path(sample_info['base_path']) / 'region_properties').glob(f'{pattern}*.csv'))
    df = pd.concat(load_rprop_file_add_sample_info(rprop_file, sample_info) for rprop_file in rprop_files).reset_index(drop=True)
    return df

# concatenate all distance files
df = pd.concat(load_all_rprop_files_for_sample(sample_info) for sample_info in res).reset_index(drop=True)
df

In [ ]:
import seaborn as sns

ax = sns.boxplot(data=df, x='sample', hue='cell_type', y='area')
ax.tick_params(axis='x', rotation=90)